# CareCaller — Whisper-small Fine-Tuning

**Self-contained notebook** — no repo clone, no file uploads needed.

**Before running:**
1. Runtime → Change runtime type → **T4 GPU** → Save
2. **Runtime → Disconnect and delete runtime → Reconnect** (do NOT use auto-connect — stale kernels won't see the GPU)
3. Open the **🔑 Secrets** panel (left sidebar) → add `HF_TOKEN` (your HuggingFace read token)
4. Accept dataset terms on HuggingFace while logged in:
   - https://huggingface.co/datasets/mozilla-foundation/common_voice_13_0
   - https://huggingface.co/datasets/openslr/librispeech_asr
   - https://huggingface.co/datasets/openslr/musan
5. Run all cells top to bottom


In [ ]:
# ── Cell 1: GPU check + install deps + HF login ───────────────────────────
import subprocess, os

# GPU check via torch (always present on Colab).
# If this warns even after selecting T4:
#   Runtime -> Disconnect and delete runtime -> reconnect (NOT auto-connect)
import torch
if torch.cuda.is_available():
    _dev  = torch.cuda.get_device_name(0)
    _vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {_dev}  VRAM: {_vram:.1f} GB  -- OK')
else:
    try:
        _r = subprocess.run(
            ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
            capture_output=True, text=True,
        )
        if _r.stdout.strip():
            print('nvidia-smi sees:', _r.stdout.strip())
    except FileNotFoundError:
        pass
    print(
        'WARNING: torch.cuda.is_available() == False.\n'
        'If you chose T4 GPU in runtime settings, the kernel is stale.\n'
        'Fix: Runtime -> Disconnect and delete runtime, then reconnect.\n'
        'Continuing anyway -- GPU-required cells will fail later.'
    )

# Install all deps
!pip install -q transformers>=4.40.0 datasets>=2.19.0 evaluate>=0.4.1 accelerate>=0.29.0 jiwer>=3.0.3 librosa>=0.10.1 soundfile>=0.12.1 scipy>=1.11.0 numpy>=1.24.0 edge-tts>=6.1.9 faster-whisper>=1.0.3 nest_asyncio>=1.6.0

# Patch Jupyter event loop so asyncio.run() works inside notebooks
import nest_asyncio; nest_asyncio.apply()

# HuggingFace login (add HF_TOKEN to Colab Secrets -> key icon in left sidebar)
from google.colab import userdata
from huggingface_hub import login
hf_token = userdata.get('HF_TOKEN')
os.environ['HF_TOKEN'] = hf_token
login(token=hf_token, add_to_git_credential=False)
print('HuggingFace login OK')


In [ ]:
# ── Cell 2: Create module directory structure on this Colab VM ─────────────
import os
for d in ['/content/stt/data', '/content/stt/train',
          '/content/stt/eval/test_sets', '/content/stt/serve',
          '/content/stt/results', '/content/stt/data/musan_cache']:
    os.makedirs(d, exist_ok=True)
    if not d.endswith('musan_cache'):
        open(f'{d}/__init__.py', 'a').close()
os.chdir('/content/stt')
print('Dirs ready:', os.getcwd())


In [ ]:
%%writefile /content/stt/data/augment.py
"""
Audio augmentation for telephony simulation.
All functions operate on float32 numpy arrays normalised to [-1, 1].
Input/output sample rate is 16 000 Hz (Whisper's expected rate).
"""
import os
import numpy as np
import librosa
from scipy import signal as _signal

# ── Noise cache ───────────────────────────────────────────────────────────────
_NOISE_CACHE: list = []

# Tried in order — first success wins.  All are Parquet (no trust_remote_code).
_NOISE_DATASET_CANDIDATES = [
    # (dataset_id,              config,    split,   label_filter)
    ('Myrtle/CAIMAN-ASR-BackgroundNoise', None,  'train', None),   # 16kHz mono, 533 MB
    ('huseinzol05/noise-dataset',         None,  'train', None),   # pure noise, 1.07 GB
    ('FluidInference/musan',              None,  'train', 'noise'),  # full MUSAN, filter noise label
]


def load_musan_noise(cache_dir: str = './data/musan_cache', max_files: int = 200) -> int:
    """
    Populate the in-process noise cache with real environmental-noise files.

    Priority:
      1. Load pre-saved .npy files from cache_dir  (fast, no network)
      2. Try HuggingFace noise datasets (no trust_remote_code)
      3. Silent fallback — pink / brown / bandlimited synthetic noise

    Returns number of files loaded (0 = synthetic fallback active).
    """
    global _NOISE_CACHE
    if _NOISE_CACHE:
        return len(_NOISE_CACHE)

    os.makedirs(cache_dir, exist_ok=True)
    npy_files = sorted(f for f in os.listdir(cache_dir) if f.endswith('.npy'))

    if npy_files:
        for fname in npy_files[:max_files]:
            arr = np.load(os.path.join(cache_dir, fname))
            if arr.shape[0] >= 16000:
                _NOISE_CACHE.append(arr.astype(np.float32))
        print(f'Noise cache: {len(_NOISE_CACHE)} files from {cache_dir}')
        return len(_NOISE_CACHE)

    try:
        from datasets import load_dataset, Audio as HFAudio
        for ds_id, config, split, label_filter in _NOISE_DATASET_CANDIDATES:
            try:
                print(f'Trying noise dataset: {ds_id} ...')
                ds = load_dataset(ds_id, config, split=split, streaming=True) \
                     if config else \
                     load_dataset(ds_id, split=split, streaming=True)
                ds = ds.cast_column('audio', HFAudio(sampling_rate=16000))
                if label_filter:
                    ds = ds.filter(lambda x: str(x.get('label', '')).lower() == label_filter)
                count = 0
                for sample in ds:
                    audio = sample['audio']['array'].astype(np.float32)
                    if audio.shape[0] < 16000:
                        continue
                    _NOISE_CACHE.append(audio)
                    np.save(os.path.join(cache_dir, f'noise_{count:04d}.npy'), audio)
                    count += 1
                    if count >= max_files:
                        break
                if count > 0:
                    print(f'Noise cache: {count} files from {ds_id} -> {cache_dir}')
                    return count
            except Exception as exc:
                print(f'  {ds_id} unavailable: {exc}')
                continue
    except ImportError:
        pass

    print('Real-world noise unavailable — synthetic fallback active '
          '(pink + brown + bandlimited; better than white noise).')
    return 0


def _get_noise_sample(length: int) -> np.ndarray:
    """
    Return a real noise segment from cache, or synthetic fallback.
    Synthetic fallback randomly picks from three noise colours for variety.
    """
    if _NOISE_CACHE:
        noise = _NOISE_CACHE[np.random.randint(len(_NOISE_CACHE))]
        if noise.shape[0] < length:
            noise = np.tile(noise, int(np.ceil(length / noise.shape[0])))
        start = np.random.randint(0, noise.shape[0] - length + 1) \
                if noise.shape[0] > length else 0
        return noise[start:start + length]

    # Synthetic fallback: random noise colour
    choice = np.random.randint(3)
    if choice == 0:
        return _pink_noise(length)
    elif choice == 1:
        return _brown_noise(length)
    else:
        return _bandlimited_noise(length)


def _pink_noise(length: int) -> np.ndarray:
    """1/f pink noise — approximates cafeteria/street background."""
    white = np.fft.rfft(np.random.randn(length).astype(np.float32))
    freqs = np.fft.rfftfreq(length)
    freqs[0] = 1.0
    pink = np.fft.irfft(white / np.sqrt(freqs), n=length).astype(np.float32)
    return pink / (np.max(np.abs(pink)) + 1e-10) * 0.04


def _brown_noise(length: int) -> np.ndarray:
    """1/f² brown noise — approximates HVAC, engine rumble, low hum."""
    white = np.fft.rfft(np.random.randn(length).astype(np.float32))
    freqs = np.fft.rfftfreq(length)
    freqs[0] = 1.0
    brown = np.fft.irfft(white / freqs, n=length).astype(np.float32)
    return brown / (np.max(np.abs(brown)) + 1e-10) * 0.04


def _bandlimited_noise(length: int, sr: int = 16000) -> np.ndarray:
    """Random-band noise — simulates line interference, hold music bleed."""
    lo = np.random.uniform(300, 1500)
    hi = np.random.uniform(lo + 200, 3400)
    nyq = sr / 2.0
    b, a = _signal.butter(4, [lo / nyq, hi / nyq], btype='band')
    noise = _signal.lfilter(b, a, np.random.randn(length)).astype(np.float32)
    return noise / (np.max(np.abs(noise)) + 1e-10) * 0.04


# ── Telephony simulation ──────────────────────────────────────────────────────

def simulate_telephony(audio: np.ndarray, sr: int) -> np.ndarray:
    """
    G.711 telephony chain:
      1. Bandpass 300-3 400 Hz  (telephone passband)
      2. Resample to 8 kHz
      3. μ-law encode + quantise + decode
      4. Resample to 16 kHz
    """
    audio = audio.astype(np.float32)
    nyq = sr / 2.0
    lo = max(300.0 / nyq, 0.001)
    hi = min(3400.0 / nyq, 0.999)
    b, a = _signal.butter(4, [lo, hi], btype='band')
    audio = _signal.lfilter(b, a, audio).astype(np.float32)
    audio_8k = librosa.resample(audio, orig_sr=sr, target_sr=8000)
    audio_8k = _apply_mulaw(audio_8k)
    return librosa.resample(audio_8k, orig_sr=8000, target_sr=16000)


def _apply_mulaw(audio: np.ndarray, mu: int = 255) -> np.ndarray:
    audio = np.clip(audio, -1.0, 1.0)
    compressed = np.sign(audio) * np.log1p(mu * np.abs(audio)) / np.log1p(mu)
    quantized = np.round(compressed * 127).astype(np.int8)
    expanded = (
        np.sign(quantized)
        * (np.exp(np.abs(quantized.astype(np.float32) / 127) * np.log1p(mu)) - 1)
        / mu
    )
    return expanded.astype(np.float32)


# ── Noise mixing ──────────────────────────────────────────────────────────────

def add_noise(audio: np.ndarray, noise: np.ndarray, snr_db: float) -> np.ndarray:
    """Mix noise into audio at the given SNR (RMS-correct)."""
    audio_rms = np.sqrt(np.mean(audio ** 2) + 1e-10)
    noise_rms = np.sqrt(np.mean(noise ** 2) + 1e-10)
    target_noise_rms = audio_rms / (10 ** (snr_db / 20.0))
    noise_scaled = noise * (target_noise_rms / noise_rms)
    if noise_scaled.shape[0] < audio.shape[0]:
        noise_scaled = np.tile(noise_scaled,
                               int(np.ceil(audio.shape[0] / noise_scaled.shape[0])))
    return np.clip(audio + noise_scaled[:audio.shape[0]], -1.0, 1.0)


def add_real_noise(audio: np.ndarray, snr_db: float) -> np.ndarray:
    """Add real noise (or synthetic fallback) at the given SNR."""
    return add_noise(audio, _get_noise_sample(audio.shape[0]), snr_db)


def white_noise(length: int, amplitude: float = 0.005) -> np.ndarray:
    """Kept for backward compat. Prefer add_real_noise()."""
    return (np.random.randn(length) * amplitude).astype(np.float32)


# ── Speech distortions ────────────────────────────────────────────────────────

def speed_perturb(audio: np.ndarray, rate: float) -> np.ndarray:
    return librosa.effects.time_stretch(audio, rate=rate)


def apply_augment_pipeline(audio: np.ndarray, sr: int = 16000,
                            snr_db: float = None, rate: float = 1.0) -> np.ndarray:
    """Full telephony + speed + noise pipeline."""
    audio = simulate_telephony(audio, sr)
    if rate != 1.0:
        audio = speed_perturb(audio, rate=rate)
    if snr_db is not None:
        audio = add_real_noise(audio, snr_db)
    return audio


In [ ]:
%%writefile /content/stt/data/synthetic.py
"""
Synthetic medical audio generation using edge-tts.
Produces .wav files at 16 kHz with paired transcripts.
Numeric confusion pairs are overrepresented — these are the hardest
category in the evaluation rubric and most absent from public datasets.
"""
import asyncio, os, random, json
from pathlib import Path
import librosa, numpy as np, soundfile as sf, edge_tts
from data.augment import simulate_telephony, add_real_noise, apply_augment_pipeline

VOICES = [
    'en-US-JennyNeural', 'en-US-GuyNeural', 'en-GB-SoniaNeural',
    'en-AU-NatashaNeural', 'en-US-AriaNeural', 'en-CA-LiamNeural',
    'en-GB-RyanNeural', 'en-US-EricNeural', 'en-IN-NeerjaNeural',
    'en-NZ-MitchellNeural',
]

DRUGS = [
    ('warfarin',      'Coumadin',    'five milligrams',           'once a day'),
    ('lisinopril',    'Zestril',     'ten milligrams',            'once daily'),
    ('metformin',     'Glucophage',  'five hundred milligrams',   'twice a day'),
    ('atorvastatin',  'Lipitor',     'twenty milligrams',         'at bedtime'),
    ('amlodipine',    'Norvasc',     'five milligrams',           'once a day'),
    ('sertraline',    'Zoloft',      'fifty milligrams',          'once daily'),
    ('gabapentin',    'Neurontin',   'three hundred milligrams',  'three times a day'),
    ('omeprazole',    'Prilosec',    'twenty milligrams',         'once daily'),
    ('furosemide',    'Lasix',       'forty milligrams',          'once a day'),
    ('escitalopram',  'Lexapro',     'ten milligrams',            'once daily'),
    ('metoprolol',    'Lopressor',   'twenty five milligrams',    'twice daily'),
    ('losartan',      'Cozaar',      'fifty milligrams',          'once a day'),
    ('levothyroxine', 'Synthroid',   'fifty micrograms',          'once daily'),
    ('albuterol',     'Ventolin',    'two point five milligrams', 'as needed'),
    ('prednisone',    'Deltasone',   'ten milligrams',            'once daily'),
    ('hydrocodone',   'Vicodin',     'five milligrams',           'every six hours'),
    ('alprazolam',    'Xanax',       'zero point five milligrams','as needed'),
    ('atenolol',      'Tenormin',    'fifty milligrams',          'once daily'),
    ('clonazepam',    'Klonopin',    'one milligram',             'twice daily'),
    ('cyclobenzaprine','Flexeril',   'ten milligrams',            'three times a day'),
]

DRUG_TEMPLATES = [
    'I take {name} {dose} {frequency}',
    'My doctor prescribed {name} {dose}',
    'I need a refill for {name}',
    'I have been on {name} {dose} for two years',
    'Can I get more {name}',
    'My {name} prescription is running out',
    'I started taking {name} {dose} last month',
    'I am on {name} {dose} {frequency}',
    'I ran out of {name}',
    'I take {name} every day',
    'My {name} dose was recently changed to {dose}',
    'The doctor increased my {name} to {dose}',
]

# ── Numeric confusion pairs ──────────────────────────────────────────────────
# These are the most evaluation-critical samples; heavily represented.
NUMERIC_CONFUSION_PAIRS = [
    ('take fifteen milligrams',          'take fifteen milligrams'),
    ('take fifty milligrams',            'take fifty milligrams'),
    ('fourteen units of insulin',        'fourteen units of insulin'),
    ('forty units of insulin',           'forty units of insulin'),
    ('nineteen milligrams',              'nineteen milligrams'),
    ('ninety milligrams',                'ninety milligrams'),
    ('eighteen tablets',                 'eighteen tablets'),
    ('eighty tablets',                   'eighty tablets'),
    ('thirteen milligrams',              'thirteen milligrams'),
    ('thirty milligrams',                'thirty milligrams'),
    ('fifteen units of insulin',         'fifteen units of insulin'),
    ('fifty units of insulin',           'fifty units of insulin'),
    ('pain is nine out of ten',          'pain is nine out of ten'),
    ('pain is five out of ten',          'pain is five out of ten'),
    ('pain is nineteen out of ten',      'pain is nineteen out of ten'),
    ('pain is ninety out of ten',        'pain is ninety out of ten'),
    ('take fourteen milligrams daily',   'take fourteen milligrams daily'),
    ('take forty milligrams daily',      'take forty milligrams daily'),
    ('sixteen milligrams',               'sixteen milligrams'),
    ('sixty milligrams',                 'sixty milligrams'),
    ('seventeen milligrams',             'seventeen milligrams'),
    ('seventy milligrams',               'seventy milligrams'),
    ('one hundred fifteen milligrams',   'one hundred fifteen milligrams'),
    ('one hundred fifty milligrams',     'one hundred fifty milligrams'),
]

SYMPTOM_TEMPLATES = [
    'My pain is about {n} out of ten',
    'I have had this symptom for {n} days',
    'I missed my {name} dose this morning',
    'I stopped taking {name} last week',
    'I have chest pain',
    'No chest pain today',
    'I feel dizzy when I stand up',
    'My blood pressure reading was {n} over {m}',
    'I have been feeling nauseous since I started {name}',
    'I am having trouble sleeping',
    'My heart is racing',
    'I feel short of breath after walking',
    'I have swelling in my ankles',
    'I have been losing weight without trying',
    'I have a persistent cough',
]


async def _synthesize(text: str, voice: str, path: str) -> None:
    await edge_tts.Communicate(text, voice).save(path)


def _tts_to_wav(text: str, voice: str, output_path: str,
                snr_db: float = None, rate: float = 1.0) -> None:
    mp3 = output_path.replace('.wav', '_tmp.mp3')
    asyncio.run(_synthesize(text, voice, mp3))
    audio, sr = librosa.load(mp3, sr=16000, mono=True)
    os.remove(mp3)
    audio = apply_augment_pipeline(audio.astype(np.float32), sr, snr_db=snr_db, rate=rate)
    sf.write(output_path, audio, 16000)


def generate_dataset(output_dir: str) -> list:
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    samples = []
    snr_choices = [None, 20.0, 15.0, 10.0, 8.0]
    rate_choices = [0.9, 1.0, 1.0, 1.1, 1.2]

    # Drug utterances
    for drug, brand, dose, freq in DRUGS:
        for template in DRUG_TEMPLATES:
            for name in [drug, brand]:
                text = template.format(name=name, dose=dose, frequency=freq)
                for voice in random.sample(VOICES, 2):
                    snr = random.choice(snr_choices)
                    rate = random.choice(rate_choices)
                    uid = abs(hash(text + voice + str(snr) + str(rate))) % 10_000_000
                    path = os.path.join(output_dir, f'drug_{uid}.wav')
                    if not os.path.exists(path):
                        _tts_to_wav(text, voice, path, snr, rate)
                    samples.append({'audio': path, 'text': text})

    # Numeric confusion pairs — 3 voices each for density
    for text, ref in NUMERIC_CONFUSION_PAIRS:
        for voice in random.sample(VOICES, 3):
            snr = random.choice(snr_choices)
            uid = abs(hash(text + voice + str(snr))) % 10_000_000
            path = os.path.join(output_dir, f'numeric_{uid}.wav')
            if not os.path.exists(path):
                _tts_to_wav(text, voice, path, snr)
            samples.append({'audio': path, 'text': ref})

    # Symptom utterances
    for template in SYMPTOM_TEMPLATES:
        for drug, _, _, _ in DRUGS[:8]:
            text = template.format(
                n=random.randint(1, 10), m=random.randint(60, 90), name=drug)
            for voice in random.sample(VOICES, 2):
                snr = random.choice(snr_choices)
                uid = abs(hash(text + voice + str(snr))) % 10_000_000
                path = os.path.join(output_dir, f'symptom_{uid}.wav')
                if not os.path.exists(path):
                    _tts_to_wav(text, voice, path, snr)
                samples.append({'audio': path, 'text': text})

    manifest_path = os.path.join(output_dir, 'manifest.json')
    with open(manifest_path, 'w') as f:
        json.dump(samples, f, indent=2)
    print(f'Generated {len(samples)} synthetic samples -> {manifest_path}')
    return samples


In [ ]:
%%writefile /content/stt/data/prepare.py
"""
Stream and preprocess training datasets from HuggingFace.
Uses streaming mode — no full downloads needed.

Changes vs baseline:
  * add_real_noise() (MUSAN / pink noise) instead of white_noise()
  * LibriSpeech train-other-500 subset for acoustic diversity
  * CommonVoice validation split for accent diversity
  * load_synthetic() oversample=4 to counter medical data scarcity
"""
import random
from typing import Iterator
import numpy as np
from datasets import load_dataset, Audio
from data.augment import (
    apply_augment_pipeline, load_musan_noise,
    simulate_telephony, speed_perturb, add_real_noise,
)

SAMPLE_RATE  = 16000
SPEED_RATES  = [0.85, 0.9, 1.0, 1.0, 1.1, 1.2, 1.3]
SNR_CHOICES  = [None, 20.0, 15.0, 10.0, 8.0, 5.0]


def _normalize(raw: dict) -> dict:
    audio = raw['audio']['array'].astype(np.float32)
    sr    = raw['audio']['sampling_rate']
    text  = (raw.get('text') or raw.get('sentence') or raw.get('transcript') or '').strip()
    return {'audio': audio, 'sr': sr, 'text': text}


def _augment(audio: np.ndarray, sr: int,
             snr_db: float = None, rate: float = 1.0) -> np.ndarray:
    return apply_augment_pipeline(audio, sr, snr_db=snr_db, rate=rate)


# ── LibriSpeech (clean + other for diversity) ─────────────────────────────

def stream_librispeech(max_samples: int = 6000) -> Iterator[dict]:
    """Stream train-clean-100 (5k) + train-other-500 (1k) for acoustic diversity."""
    plan = [
        ('train.clean.100', 5000),
        ('train.other.500', max_samples - 5000),
    ]
    for split, n in plan:
        if n <= 0:
            continue
        try:
            ds = load_dataset('openslr/librispeech_asr', 'clean' if 'clean' in split else 'other',
                              split=split, streaming=True, trust_remote_code=True)
            ds = ds.cast_column('audio', Audio(sampling_rate=SAMPLE_RATE))
            count = 0
            for raw in ds:
                if count >= n:
                    break
                s = _normalize(raw)
                if not s['text']:
                    continue
                yield {'audio': _augment(s['audio'], s['sr'],
                                         random.choice(SNR_CHOICES),
                                         random.choice(SPEED_RATES)),
                       'text': s['text']}
                count += 1
            print(f'LibriSpeech {split}: {count} samples')
        except Exception as exc:
            print(f'LibriSpeech {split} unavailable ({exc}), skipping')


# ── CommonVoice (validated en + accented diversity) ───────────────────────

def stream_commonvoice(max_samples: int = 4000) -> Iterator[dict]:
    """Validated English — inherently multi-accent (contributors worldwide)."""
    ds = load_dataset('mozilla-foundation/common_voice_13_0', 'en',
                      split='train', streaming=True, trust_remote_code=True)
    ds = ds.cast_column('audio', Audio(sampling_rate=SAMPLE_RATE))
    count = 0
    for raw in ds:
        if count >= max_samples:
            break
        s = _normalize(raw)
        if not s['text']:
            continue
        yield {'audio': _augment(s['audio'], s['sr'],
                                  random.choice(SNR_CHOICES),
                                  random.choice(SPEED_RATES)),
               'text': s['text']}
        count += 1
    print(f'CommonVoice en validated: {count} samples')


def stream_commonvoice_accented(max_samples: int = 1000) -> Iterator[dict]:
    """
    CommonVoice 'other' split — unvalidated but maximally diverse accents.
    Used to supplement training with non-native English speakers.
    """
    try:
        ds = load_dataset('mozilla-foundation/common_voice_13_0', 'en',
                          split='other', streaming=True, trust_remote_code=True)
        ds = ds.cast_column('audio', Audio(sampling_rate=SAMPLE_RATE))
        count = 0
        for raw in ds:
            if count >= max_samples:
                break
            s = _normalize(raw)
            if not s['text']:
                continue
            yield {'audio': _augment(s['audio'], s['sr'],
                                      random.choice(SNR_CHOICES),
                                      random.choice(SPEED_RATES)),
                   'text': s['text']}
            count += 1
        print(f'CommonVoice accented (other): {count} samples')
    except Exception as exc:
        print(f'CommonVoice accented unavailable ({exc}), skipping')


# ── Synthetic medical (oversampled) ──────────────────────────────────────

def load_synthetic(manifest_path: str, oversample: int = 4) -> Iterator[dict]:
    """
    Load synthetic medical audio.  Oversampled x{oversample} to counter the
    medical-data scarcity vs general-speech imbalance.  Each copy receives an
    independent random augmentation so the model sees varied versions.
    """
    import json, librosa
    with open(manifest_path) as f:
        samples = json.load(f)
    expanded = samples * oversample
    random.shuffle(expanded)
    loaded = 0
    for s in expanded:
        try:
            audio, _ = librosa.load(s['audio'], sr=SAMPLE_RATE, mono=True)
            snr  = random.choice([None, 20.0, 15.0, 10.0, 8.0])
            rate = random.choice([0.9, 1.0, 1.0, 1.1])
            audio = _augment(audio.astype(np.float32), SAMPLE_RATE, snr, rate)
            yield {'audio': audio, 'text': s['text']}
            loaded += 1
        except Exception:
            continue
    print(f'Synthetic (x{oversample}): {loaded} samples')


In [ ]:
%%writefile /content/stt/train/fine_tune.py
"""
Fine-tune openai/whisper-small on telephony + medical audio.
Run from /content/stt/: python train/fine_tune.py

Key settings vs baseline:
  * suppress_tokens=[]  — prevents Whisper suppressing rare drug-name tokens
  * apply_spec_augment  — time+freq masking for robustness
  * max_steps=4000      — doubled training budget
  * gradient_accumulation_steps=4  — effective batch 32
  * lr_scheduler_type=cosine        — smoother convergence tail
  * MUSAN real noise loaded before dataset build
  * Synthetic data streamed with oversample=4
"""
import sys, os, random, argparse
sys.path.insert(0, '/content/stt')

import numpy as np
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Union

import evaluate
from datasets import Dataset
from transformers import (
    WhisperForConditionalGeneration, WhisperProcessor,
    Seq2SeqTrainingArguments, Seq2SeqTrainer,
)
from data.augment   import load_musan_noise
from data.prepare   import (stream_librispeech, stream_commonvoice,
                             stream_commonvoice_accented, load_synthetic)
from data.synthetic import generate_dataset

MODEL_NAME     = 'openai/whisper-small'
SAMPLE_RATE    = 16000
CHECKPOINT_DIR = './checkpoints'
SYNTHETIC_DIR  = './data/synthetic_audio'
MUSAN_CACHE    = './data/musan_cache'

# Medical context prompt used at inference time (not forced during training,
# but registered here so benchmark.py and modal_app.py use the same string).
MEDICAL_PROMPT = (
    'Medical phone call transcription. Patient discussing medications, '
    'symptoms, dosages, side effects, and adherence. '
    'Common drugs: warfarin, lisinopril, metformin, atorvastatin, amlodipine, '
    'sertraline, gabapentin, omeprazole, furosemide, escitalopram, '
    'metoprolol, losartan, levothyroxine, albuterol, prednisone.'
)

processor  = WhisperProcessor.from_pretrained(MODEL_NAME, language='English', task='transcribe')
wer_metric = evaluate.load('wer')


@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{'input_features': f['input_features']} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors='pt')
        label_features = [{'input_ids': f['labels']} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors='pt')
        labels = labels_batch['input_ids'].masked_fill(labels_batch.attention_mask.ne(1), -100)
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]
        batch['labels'] = labels
        return batch


def prepare_sample(sample: dict) -> dict:
    audio = sample['audio']
    text  = sample['text'].strip()
    if len(text) < 2:
        return None
    if audio.shape[0] > 480_000:
        audio = audio[:480_000]
    feats  = processor.feature_extractor(audio, sampling_rate=SAMPLE_RATE,
                                          return_tensors='np').input_features[0]
    labels = processor.tokenizer(text).input_ids
    return {'input_features': feats, 'labels': labels}


def compute_metrics(pred):
    pred_ids, label_ids = pred.predictions, pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str  = processor.tokenizer.batch_decode(pred_ids,  skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    return {'wer': round(100 * wer_metric.compute(predictions=pred_str, references=label_str), 2)}


def build_dataset():
    all_raw = []

    print('Streaming LibriSpeech (clean-100 + other-500)...')
    all_raw.extend(list(stream_librispeech(max_samples=6000)))

    print('Streaming CommonVoice validated...')
    all_raw.extend(list(stream_commonvoice(max_samples=4000)))

    print('Streaming CommonVoice accented (other split)...')
    all_raw.extend(list(stream_commonvoice_accented(max_samples=1000)))

    print('Generating synthetic medical audio...')
    generate_dataset(SYNTHETIC_DIR)
    all_raw.extend(list(load_synthetic(f'{SYNTHETIC_DIR}/manifest.json', oversample=4)))

    print(f'Total raw samples: {len(all_raw)}')
    random.shuffle(all_raw)

    processed = [r for s in all_raw if (r := prepare_sample(s)) is not None]
    print(f'Valid processed samples: {len(processed)}')

    split = int(0.95 * len(processed))
    return Dataset.from_list(processed[:split]), Dataset.from_list(processed[split:])


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--resume_from_checkpoint', type=str, default=None)
    args = parser.parse_args()

    print(f'CUDA: {torch.cuda.is_available()}')
    if torch.cuda.is_available():
        print(f'GPU: {torch.cuda.get_device_name(0)}'
              f'  VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

    # Load MUSAN noise cache before dataset build (subprocesses need this)
    n_noise = load_musan_noise(MUSAN_CACHE, max_files=200)
    print(f'Noise cache: {n_noise} files  ({"MUSAN" if n_noise else "pink-noise fallback"})')

    model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)

    # ── Critical fine-tuning settings ────────────────────────────────────
    # suppress_tokens=[] prevents Whisper from suppressing rare token IDs
    # (drug names are often rare in vocab → suppressed by default → wrong).
    model.config.suppress_tokens      = []
    model.config.begin_suppress_tokens = []
    model.generation_config.suppress_tokens      = []
    model.generation_config.begin_suppress_tokens = []

    # SpecAugment: time + frequency masking during training
    model.config.apply_spec_augment = True
    model.config.mask_time_prob     = 0.05
    model.config.mask_feature_prob  = 0.05

    model.generation_config.language          = 'english'
    model.generation_config.task              = 'transcribe'
    model.generation_config.forced_decoder_ids = None

    train_ds, eval_ds = build_dataset()

    training_args = Seq2SeqTrainingArguments(
        output_dir                  = CHECKPOINT_DIR,
        per_device_train_batch_size = 8,
        per_device_eval_batch_size  = 8,
        gradient_accumulation_steps = 4,       # effective batch 32
        learning_rate               = 1e-5,
        lr_scheduler_type           = 'cosine',
        warmup_steps                = 500,
        max_steps                   = 4000,
        gradient_checkpointing      = True,
        fp16                        = True,
        eval_strategy               = 'steps',
        predict_with_generate       = True,
        generation_max_length       = 225,     # Whisper natural max
        save_steps                  = 500,
        eval_steps                  = 500,
        logging_steps               = 25,
        load_best_model_at_end      = True,
        metric_for_best_model       = 'wer',
        greater_is_better           = False,
        push_to_hub                 = False,
        dataloader_num_workers      = 2,
        remove_unused_columns       = False,
        label_names                 = ['labels'],
    )

    trainer = Seq2SeqTrainer(
        args             = training_args,
        model            = model,
        train_dataset    = train_ds,
        eval_dataset     = eval_ds,
        data_collator    = DataCollatorSpeechSeq2SeqWithPadding(processor=processor),
        compute_metrics  = compute_metrics,
        processing_class = processor.feature_extractor,
    )

    print('Starting training...')
    trainer.train(resume_from_checkpoint=args.resume_from_checkpoint)

    final_dir = f'{CHECKPOINT_DIR}/whisper-telephony-medical-final'
    trainer.save_model(final_dir)
    processor.save_pretrained(final_dir)
    print(f'Model saved -> {final_dir}')


if __name__ == '__main__':
    main()


In [ ]:
%%writefile /content/stt/eval/test_sets/generate.py
"""
Generate evaluation test sets.
Run: cd /content/stt && python eval/test_sets/generate.py

Sets built:
  clean_8k          — LibriSpeech test-clean downsampled to 8 kHz
  noisy_8k          — above + MUSAN/pink noise at 10 dB SNR
  medical_terms     — TTS drug sentences at 8 kHz + noise
  numeric_confusion — TTS fifteen/fifty confusion pairs
  accented          — CommonVoice test split (multi-accent, 8 kHz)
"""
import asyncio, json, os, sys
sys.path.insert(0, '/content/stt')
import librosa, numpy as np, soundfile as sf
from data.augment import simulate_telephony, add_real_noise, load_musan_noise

VOICES = ['en-US-JennyNeural', 'en-US-GuyNeural', 'en-GB-SoniaNeural',
          'en-AU-NatashaNeural', 'en-IN-NeerjaNeural', 'en-NZ-MitchellNeural']

NUMERIC_CONFUSION = [
    ('take fifteen milligrams',            'take fifteen milligrams'),
    ('take fifty milligrams',              'take fifty milligrams'),
    ('fourteen units of insulin',          'fourteen units of insulin'),
    ('forty units of insulin',             'forty units of insulin'),
    ('nineteen milligrams daily',          'nineteen milligrams daily'),
    ('ninety milligrams daily',            'ninety milligrams daily'),
    ('eighteen tablets a week',            'eighteen tablets a week'),
    ('eighty tablets a week',              'eighty tablets a week'),
    ('pain level nine out of ten',         'pain level nine out of ten'),
    ('pain level five out of ten',         'pain level five out of ten'),
    ('thirteen milligrams twice a day',    'thirteen milligrams twice a day'),
    ('thirty milligrams twice a day',      'thirty milligrams twice a day'),
    ('sixteen milligrams once daily',      'sixteen milligrams once daily'),
    ('sixty milligrams once daily',        'sixty milligrams once daily'),
    ('seventeen units',                    'seventeen units'),
    ('seventy units',                      'seventy units'),
]

MEDICAL_TERMS = [
    'I take warfarin five milligrams once a day',
    'My lisinopril prescription needs a refill',
    'I have been on metformin five hundred milligrams for two years',
    'The atorvastatin is causing muscle pain',
    'I need more amlodipine',
    'I take sertraline fifty milligrams every morning',
    'My gabapentin dose was recently increased',
    'I ran out of omeprazole',
    'Furosemide forty milligrams once daily',
    'Escitalopram ten milligrams for anxiety',
    'I take metoprolol twenty five milligrams twice daily',
    'My Lexapro prescription is empty',
    'Lasix forty milligrams in the morning',
    'I started Zoloft last week',
    'My Lipitor causes leg cramps',
]


async def _tts_mp3(text: str, voice: str, path: str) -> None:
    import edge_tts
    await edge_tts.Communicate(text, voice).save(path)


def _make_wav(text: str, voice: str, path: str, snr_db: float = None) -> None:
    mp3 = path.replace('.wav', '_tmp.mp3')
    asyncio.run(_tts_mp3(text, voice, mp3))
    audio, sr = librosa.load(mp3, sr=16000, mono=True)
    os.remove(mp3)
    audio = simulate_telephony(audio.astype(np.float32), sr)
    if snr_db is not None:
        audio = add_real_noise(audio, snr_db)
    sf.write(path, audio, 16000)


def _generate_accented(output_dir: str, max_samples: int = 60) -> list:
    """Pull CommonVoice test split for a real accented speech test set."""
    samples = []
    acc_dir = os.path.join(output_dir, 'accented')
    os.makedirs(acc_dir, exist_ok=True)
    try:
        from datasets import load_dataset, Audio
        ds = load_dataset('mozilla-foundation/common_voice_13_0', 'en',
                          split='test', streaming=True, trust_remote_code=True)
        ds = ds.cast_column('audio', Audio(sampling_rate=16000))
        count = 0
        for raw in ds:
            if count >= max_samples:
                break
            text = raw.get('sentence', '').strip()
            if not text:
                continue
            audio = raw['audio']['array'].astype(np.float32)
            audio = simulate_telephony(audio, 16000)
            path  = os.path.join(acc_dir, f'acc_{count:03d}.wav')
            sf.write(path, audio, 16000)
            samples.append({'audio': path, 'text': text})
            count += 1
        print(f'Accented (CommonVoice test): {count} samples')
    except Exception as exc:
        print(f'Accented set skipped ({exc})')
    return samples


def generate(output_dir: str = None) -> dict:
    if output_dir is None:
        output_dir = os.path.join(os.path.dirname(__file__), 'test_audio')

    load_musan_noise('./data/musan_cache')

    sets = {k: [] for k in ['clean_8k', 'noisy_8k', 'medical_terms',
                              'numeric_confusion', 'accented']}
    for s in sets:
        os.makedirs(os.path.join(output_dir, s), exist_ok=True)

    # Medical terms
    for i, text in enumerate(MEDICAL_TERMS):
        for vi, voice in enumerate(VOICES[:3]):
            uid  = f'med_{i}_{vi}'
            cp   = os.path.join(output_dir, 'clean_8k',      f'{uid}.wav')
            np_  = os.path.join(output_dir, 'noisy_8k',      f'{uid}.wav')
            mt   = os.path.join(output_dir, 'medical_terms',  f'{uid}.wav')
            if not os.path.exists(cp):  _make_wav(text, voice, cp)
            if not os.path.exists(np_): _make_wav(text, voice, np_, snr_db=10.0)
            if not os.path.exists(mt):
                import shutil; shutil.copy(cp, mt)
            sets['clean_8k'].append({'audio': cp,  'text': text})
            sets['noisy_8k'].append({'audio': np_, 'text': text})
            sets['medical_terms'].append({'audio': mt, 'text': text})

    # Numeric confusion
    for i, (text, ref) in enumerate(NUMERIC_CONFUSION):
        for vi, voice in enumerate(VOICES[:3]):
            uid  = f'num_{i}_{vi}'
            path = os.path.join(output_dir, 'numeric_confusion', f'{uid}.wav')
            if not os.path.exists(path):
                _make_wav(text, voice, path, snr_db=8.0)
            sets['numeric_confusion'].append({'audio': path, 'text': ref})

    # Accented (real CommonVoice)
    sets['accented'] = _generate_accented(output_dir, max_samples=60)

    manifest_path = os.path.join(output_dir, 'manifest.json')
    with open(manifest_path, 'w') as f:
        json.dump(sets, f, indent=2)
    for name, items in sets.items():
        print(f'  {name}: {len(items)} samples')
    print(f'Manifest -> {manifest_path}')
    return sets


if __name__ == '__main__':
    print('Generating evaluation test sets...')
    generate()


In [ ]:
%%writefile /content/stt/eval/benchmark.py
"""
STT Evaluation Benchmark.
Usage: cd /content/stt && python eval/benchmark.py [./checkpoints/whisper-telephony-medical-final]

Improvements vs baseline:
  * initial_prompt (medical context) used for all model calls — shows its effect
  * Fuzzy drug-name recall (Levenshtein distance ≤ 2) — catches partial transcriptions
  * Accented speech set evaluated
  * Table shows both baseline and fine-tuned, with prompt and without
"""
import json, os, re, sys
import jiwer
from faster_whisper import WhisperModel

sys.path.insert(0, '/content/stt')
from train.fine_tune import MEDICAL_PROMPT

KNOWN_DRUGS = [
    'warfarin', 'coumadin', 'lisinopril', 'zestril', 'metformin', 'glucophage',
    'atorvastatin', 'lipitor', 'amlodipine', 'norvasc', 'sertraline', 'zoloft',
    'gabapentin', 'neurontin', 'omeprazole', 'prilosec', 'furosemide', 'lasix',
    'escitalopram', 'lexapro', 'metoprolol', 'lopressor', 'losartan', 'cozaar',
    'levothyroxine', 'synthroid', 'albuterol', 'ventolin', 'prednisone',
    'hydrocodone', 'vicodin', 'alprazolam', 'xanax', 'atenolol', 'tenormin',
    'clonazepam', 'klonopin', 'cyclobenzaprine', 'flexeril',
]

NORM = jiwer.Compose([jiwer.ToLowerCase(), jiwer.RemovePunctuation(),
                       jiwer.Strip(), jiwer.ReduceToListOfListOfWords()])


def _levenshtein(a: str, b: str) -> int:
    """Compute edit distance between two strings."""
    m, n = len(a), len(b)
    dp = list(range(n + 1))
    for i in range(1, m + 1):
        prev, dp[0] = dp[0], i
        for j in range(1, n + 1):
            temp = dp[j]
            dp[j] = prev if a[i-1] == b[j-1] else 1 + min(prev, dp[j], dp[j-1])
            prev = temp
    return dp[n]


def _drug_in_text(drug: str, text: str) -> bool:
    """
    Check if a drug name appears in text.
    Accepts exact substring OR Levenshtein distance ≤ 2 on individual words
    (catches 'warfaren' → 'warfarin', 'Prilosek' → 'Prilosec').
    """
    tl = text.lower()
    dl = drug.lower()
    if dl in tl:
        return True
    for word in tl.split():
        if abs(len(word) - len(dl)) <= 2 and _levenshtein(dl, word) <= 2:
            return True
    return False


def _load_model(path: str) -> WhisperModel:
    return WhisperModel(path, device='cuda', compute_type='float16')


def _transcribe(model: WhisperModel, path: str, use_prompt: bool = True) -> str:
    prompt = MEDICAL_PROMPT if use_prompt else None
    segs, _ = model.transcribe(path, language='en',
                                word_timestamps=False, initial_prompt=prompt)
    return ' '.join(s.text.strip() for s in segs).strip()


def compute_wer(refs, hyps):
    return jiwer.wer(refs, hyps, reference_transform=NORM, hypothesis_transform=NORM)

def compute_cer(refs, hyps):
    return jiwer.cer([r.lower() for r in refs], [h.lower() for h in hyps])

def compute_med_recall(refs, hyps):
    hits, total = 0, 0
    for r, h in zip(refs, hyps):
        for d in KNOWN_DRUGS:
            if d in r.lower():
                total += 1
                hits  += int(_drug_in_text(d, h))
    return hits / max(total, 1)

def compute_num_acc(refs, hyps):
    correct, total = 0, 0
    for r, h in zip(refs, hyps):
        for n in re.findall(r'\b\d+\b', r):
            total   += 1
            correct += int(n in re.findall(r'\b\d+\b', h))
    return correct / max(total, 1)


def evaluate_set(model: WhisperModel, samples: list, use_prompt: bool = True) -> dict:
    refs = [s['text'] for s in samples]
    hyps = [_transcribe(model, s['audio'], use_prompt) for s in samples]
    return {
        'wer_pct':              round(compute_wer(refs, hyps) * 100, 2),
        'cer_pct':              round(compute_cer(refs, hyps) * 100, 2),
        'medical_recall_pct':   round(compute_med_recall(refs, hyps) * 100, 2),
        'numeric_accuracy_pct': round(compute_num_acc(refs, hyps) * 100, 2),
        'n_samples':            len(samples),
        'prompt':               use_prompt,
    }


def print_table(results: dict) -> None:
    W = 80
    print('\n' + '=' * W)
    print('  CareCaller STT Benchmark Results')
    print('=' * W)
    hdr = f'{"Set":<22} {"Model":<20} {"Prompt":>6} {"WER%":>6} {"CER%":>6} {"MedRec%":>8} {"NumAcc%":>8}'
    print(hdr)
    print('-' * W)
    for set_name, rows in results.items():
        for label, m in rows.items():
            p = 'yes' if m['prompt'] else 'no '
            print(f'{set_name:<22} {label:<20} {p:>6} '
                  f'{m["wer_pct"]:>6} {m["cer_pct"]:>6} '
                  f'{m["medical_recall_pct"]:>8} {m["numeric_accuracy_pct"]:>8}')
    print('=' * W)


def main() -> None:
    manifest_path = os.path.join(
        os.path.dirname(__file__), 'test_sets', 'test_audio', 'manifest.json')
    if not os.path.exists(manifest_path):
        print('Run: python eval/test_sets/generate.py  first')
        sys.exit(1)

    with open(manifest_path) as f:
        manifest = json.load(f)

    finetuned_path = sys.argv[1] if len(sys.argv) > 1 else None

    print('Loading baseline whisper-small...')
    baseline = _load_model('small')
    finetuned = _load_model(finetuned_path) if finetuned_path else None

    results = {}
    for set_name, samples in manifest.items():
        if not samples:
            continue
        print(f'\nEvaluating {set_name} ({len(samples)} samples)...')
        results[set_name] = {}
        # Baseline without prompt
        results[set_name]['baseline (no prompt)'] = evaluate_set(
            baseline, samples, use_prompt=False)
        # Baseline with medical prompt
        results[set_name]['baseline + prompt'] = evaluate_set(
            baseline, samples, use_prompt=True)
        if finetuned:
            # Fine-tuned with medical prompt
            results[set_name]['fine-tuned + prompt'] = evaluate_set(
                finetuned, samples, use_prompt=True)

    print_table(results)

    os.makedirs('results', exist_ok=True)
    with open('results/benchmark_results.json', 'w') as f:
        json.dump(results, f, indent=2)
    print('\nResults saved -> results/benchmark_results.json')


if __name__ == '__main__':
    main()


In [ ]:
# ── Cell 9: Load real-world noise cache ────────────────────────────────────
# Tries noise datasets in order (all Parquet, no trust_remote_code):
#   1. Myrtle/CAIMAN-ASR-BackgroundNoise  (16kHz mono, augmentation-focused, 533 MB)
#   2. huseinzol05/noise-dataset          (pure noise corpus, 1.07 GB)
#   3. FluidInference/musan               (full MUSAN, noise label filtered, 585 MB)
# Falls back to pink + brown + bandlimited synthetic noise if all fail.
# Downloaded files saved as .npy to data/musan_cache/ — reused on re-runs.
import sys, os
sys.path.insert(0, '/content/stt')
os.chdir('/content/stt')

from data.augment import load_musan_noise
n = load_musan_noise(cache_dir='./data/musan_cache', max_files=200)
if n > 0:
    print(f'\nNoise cache ready: {n} real-world noise files')
else:
    print('\nSynthetic noise fallback active (pink + brown + bandlimited).')
    print('Augmentation is still significantly better than white noise.')


In [ ]:
# ── Cell 10: Smoke test — verify all modules and augmentation pipeline ──────
import sys, os
sys.path.insert(0, '/content/stt')
os.chdir('/content/stt')
import numpy as np

from data.augment import (
    simulate_telephony, add_noise, add_real_noise, speed_perturb,
    white_noise, apply_augment_pipeline, load_musan_noise,
)

audio  = np.random.randn(16000).astype(np.float32) * 0.3

# telephony simulation (bandpass + 8kHz + μ-law)
tele = simulate_telephony(audio, 16000)
assert tele.dtype == np.float32, 'dtype'
assert tele.shape[0] > 0, 'empty'

# real noise (MUSAN or pink fallback)
noisy = add_real_noise(tele, snr_db=10.0)
assert np.max(np.abs(noisy)) <= 1.0, 'clipping'

# speed perturbation
fast = speed_perturb(audio, rate=1.2)
assert fast.shape[0] < audio.shape[0], 'speed perturb'

slow = speed_perturb(audio, rate=0.85)
assert slow.shape[0] > audio.shape[0], 'slow speed'

# full pipeline
out = apply_augment_pipeline(audio, sr=16000, snr_db=12.0, rate=1.1)
assert out.dtype == np.float32

print('All augmentation smoke tests passed')


In [ ]:
# ── Cell 11: Generate synthetic medical training audio ──────────────────────
# ~1 200 samples (expanded drug list + more numeric pairs + more voices).
# Takes ~25-35 min (edge-tts network calls). Already-generated files skipped.
import nest_asyncio; nest_asyncio.apply()
import sys, os
sys.path.insert(0, '/content/stt')
os.chdir('/content/stt')

from data.augment import load_musan_noise
load_musan_noise('./data/musan_cache')  # ensure noise cache is ready

from data.synthetic import generate_dataset
samples = generate_dataset('./data/synthetic_audio')
print(f'Done: {len(samples)} synthetic samples')


In [ ]:
# ── Cell 12: Fine-tune whisper-small (~3 hrs on T4, 4 000 steps) ───────────
# Saves checkpoint every 500 steps.
# If Colab disconnects, run Cell 13 (resume) instead of this cell.
import os
os.chdir('/content/stt')
!python train/fine_tune.py


In [ ]:
# ── Cell 13: Resume from latest checkpoint (only if Cell 12 was interrupted)
import os, glob
os.chdir('/content/stt')
checkpoints = sorted(glob.glob('./checkpoints/checkpoint-*'),
                     key=lambda p: int(p.split('-')[-1]))
if checkpoints:
    latest = checkpoints[-1]
    print(f'Resuming from: {latest}')
    !python train/fine_tune.py --resume_from_checkpoint {latest}
else:
    print('No checkpoints found — run Cell 12 first')


In [ ]:
# ── Cell 14: Generate eval test sets (skip if already done) ────────────────
import os
os.chdir('/content/stt')
manifest = 'eval/test_sets/test_audio/manifest.json'
if not os.path.exists(manifest):
    import nest_asyncio; nest_asyncio.apply()
    !python eval/test_sets/generate.py
else:
    print(f'Test sets already at {manifest}')


In [ ]:
# ── Cell 15: Benchmark — baseline vs fine-tuned (with + without prompt) ────
import os
os.chdir('/content/stt')
!python eval/benchmark.py ./checkpoints/whisper-telephony-medical-final


In [ ]:
# ── Cell 16: Zip and download checkpoint ───────────────────────────────────
import shutil
from google.colab import files

shutil.make_archive('/content/whisper-telephony-medical-final', 'zip',
                    '/content/stt/checkpoints/whisper-telephony-medical-final')
print('Zipped. Starting download...')
files.download('/content/whisper-telephony-medical-final.zip')


## After downloading — upload to Modal (run locally)

```bash
# 1. Unzip into your local stt/checkpoints/ directory
mkdir -p stt/checkpoints/whisper-telephony-medical-final
# On Windows (PowerShell):
Expand-Archive whisper-telephony-medical-final.zip -DestinationPath stt/checkpoints/whisper-telephony-medical-final

# 2. Upload to Modal volume
stt/.venv/Scripts/python.exe -m modal volume put carecaller-stt-model \
  stt/checkpoints/whisper-telephony-medical-final /finetuned

# 3. Redeploy
stt/.venv/Scripts/python.exe -m modal deploy stt/serve/modal_app.py

# 4. Set in .env:
# STT_PROVIDER=custom
# CUSTOM_STT_URL=<url from modal deploy output>
```
